In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from rdkit import Chem
from sklearn.metrics import r2_score, mean_squared_error
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

In [ ]:
# df = pd.read_parquet("../Dane/chembl_ml_dataset_04.parquet")
df = pd.read_parquet("../Dane/chembl_ml_dataset_04_CHEMBL2147.parquet")

df = df[df["standard_type"] == "IC50"]
df = df[df["pchembl_value"].notna()]
df = df[df["canonical_smiles"].notna()]
target = df["target_chembl_id"].value_counts().idxmax()
df = df[df["target_chembl_id"] == target]
df = df.drop_duplicates("canonical_smiles")
df = df.reset_index(drop=True)

print("Target:", target)
print("Samples:", len(df))

In [ ]:
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        int(atom.GetIsAromatic()),
        int(atom.GetHybridization()),
        atom.GetFormalCharge()
    ]

In [ ]:
def smiles_to_graph(smiles, y):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    # node features
    x = []
    for atom in mol.GetAtoms():
        x.append(atom_features(atom))

    x = torch.tensor(x, dtype=torch.float)

    # edges
    edge_index = []

    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    y = torch.tensor([y], dtype=torch.float)
    data = Data(x=x, edge_index=edge_index, y=y)
    return data

In [ ]:
class MoleculeDataset(Dataset):
    def __init__(self, df):
        super().__init__()
        self.graphs = []
        for _, row in df.iterrows():
            graph = smiles_to_graph(
                row["canonical_smiles"],
                row["pchembl_value"]
            )
            if graph is not None:
                self.graphs.append(graph)

    def len(self):
        return len(self.graphs)

    def get(self, idx):
        return self.graphs[idx]

# Model

In [ ]:
dataset = MoleculeDataset(df)

train_size = int(0.7 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
class GCN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=256):
        super().__init__()

        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.conv3 = GCNConv(hidden_dim, hidden_dim)

        self.lin1 = nn.Linear(hidden_dim, 64)
        self.lin2 = nn.Linear(64, 1)

        self.dropout = nn.Dropout(0.2)

    def forward(self, x, edge_index, batch):
        # graph conv
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.conv3(x, edge_index)
        x = F.relu(x)

        # pooling
        x = global_mean_pool(x, batch)

        # MLP
        x = self.lin1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.lin2(x)

        return x

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GCN().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Trening

In [ ]:
EPOCHS = 301
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_losses.append(train_loss)
    model.eval()
    total_val_loss = 0

    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y.view(-1, 1))
            total_val_loss += loss.item()

    val_loss = total_val_loss / len(test_loader)
    val_losses.append(val_loss)

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}"
    )
    if epoch % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'../Dane/Modele/gnn_checkpoint_{epoch}_30.pt')

In [ ]:
model.eval()
preds = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        preds.extend(out.cpu().numpy().flatten())
        targets.extend(batch.y.cpu().numpy().flatten())

preds = np.array(preds)
targets = np.array(targets)

r2 = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))
print("R2:", r2)
print("RMSE:", rmse)

In [ ]:
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, "gnn_checkpoint.pt")

# Wyniki

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train loss")
plt.plot(val_losses, label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training curves")
plt.legend()

plt.show()

In [ ]:
def LoadCheckpoint(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print("Continuing from epoch:", start_epoch)

checkpoint_path = "../Dane/Modele/gnn_checkpoint_300_30.pt"
# LoadCheckpoint(checkpoint_path)

# Dotrenowanie

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=30
)

In [ ]:
START_EPOCH = 301
END_EPOCH = 1201

train_losses_cont = []
val_losses_cont = []

for epoch in range(START_EPOCH, END_EPOCH):
    model.train()

    total_loss = 0

    for batch in train_loader:
        batch = batch.to(device)

        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index, batch.batch)

        loss = criterion(out, batch.y.view(-1, 1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)
    train_losses_cont.append(train_loss)

    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y.view(-1, 1))
            total_val_loss += loss.item()

    val_loss = total_val_loss / len(test_loader)
    val_losses_cont.append(val_loss)

    # scheduler
    scheduler.step(val_loss)

    # learning rate
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch}: "
        f"train_loss={train_loss:.4f}, "
        f"val_loss={val_loss:.4f}, "
        f"lr={current_lr:.6f}"
    )

    if epoch % 100 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, f'../Dane/Modele/gnn_checkpoint_{epoch}_30.pt')

In [ ]:
model.eval()
preds = []
targets = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        preds.extend(out.cpu().numpy().flatten())
        targets.extend(batch.y.cpu().numpy().flatten())

preds = np.array(preds)
targets = np.array(targets)

r2 = r2_score(targets, preds)
rmse = np.sqrt(mean_squared_error(targets, preds))

print("R2:", r2)
print("RMSE:", rmse)

In [ ]:
epoki = range(300, 300 + len(train_losses_cont))
plt.figure(figsize=(10,5))
plt.plot(epoki, train_losses_cont, label="Train loss")
plt.plot(epoki, val_losses_cont, label="Validation loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training continuation")

plt.legend()
plt.show()

In [ ]:
# torch.save(model.state_dict(), "../Dane/gnn_01.pt")